In [2]:
from pysat.formula import CNF
from pysat.solvers import Solver
from pysat.solvers import Kissat404
from pysat.solvers import Glucose42
from partitionsolver.utils import file_reader
from multiprocessing import Process, Queue
import partitionsolver.utils.hypergraph_worker as hypergraph_worker
import time
import os
import json

import numpy as np

from partitionsolver.solver.division_solver import DivisionDPLL
from partitionsolver.solver.partition_solver import PartitionDPLL
from  partitionsolver.utils import literal_util

import lzma


SyntaxError: did you forget parentheses around the comprehension target? (partition_solver.py, line 119)

In [ ]:
timings_file = "../output/timings.json"
timings_file_old = "../output/timings_old.json"

#cnf_file_location = "./instances/instances_small/d6afa5689d75e37111656db8980dd54b-grs-160-48.cnf.xz"
#cnf_file_location = "./instances/instances_small/4c3001f8073986116d98084dde70da05-fsf-300-354-2-2-3-2.9.opt.cnf.xz"
#cnf_file_location = "./instances/instances_small/ad9eb96bac59319fc2f7daffd1f961f8-AProVE07-21.cnf.xz"
#cnf_file_location = "./instances/instances_small/77a0d54f2fb3740a9a321623c0c10f3e-tseitin_grid_n12_m12.cnf.xz"
#cnf_file_location = "./instances/instances_small/b628043a07c5576dd6cd21c9d73a69e0-fixedbandwidth-eq-37_shuffled.cnf.xz"
#cnf_file_location = "./instances/custom/two_random_connect8.cnf"
cnf_file_location = "../instances/random_sat/uf50-028.cnf"

In [ ]:

stats = {}

def load_stats_file():
    global stats
    if stats is None:
        try:
            with open(timings_file) as f:
                stats = json.load(f)
        except FileNotFoundError:
            stats = {}
    return stats

def save_stats():
    global stats
    if stats is None:
        return
    
    old_stats = {}
    try:
        with open(timings_file_old) as f:
            old_stats = json.load(f)
    except FileNotFoundError:
        old_stats = {}

    with open(timings_file_old, "w") as f:
        json.dump(old_stats, f, indent=2)
    with open(timings_file, "w") as f:
        json.dump(stats, f, indent=2)

def set_stat(path, solver, satisfiable, time, variables, clauses, extradata = None):
    if path not in stats:
        stats[path] = {
            "variables": variables,
            "clauses": clauses
        }
    stats[path][solver] = {
        "satisfiable": satisfiable,
        "time": time,
        "data": extradata
    }

In [ ]:
def extract_common_variables(clausesA, clausesB, num_vars):
    maskA = [False] * num_vars
    maskB = [False] * num_vars
    for clause in clausesA:
        for lit in clause:
            maskA[abs(lit) - 1] = True
    for clause in clausesB:
        for lit in clause:
            maskB[abs(lit) - 1] = True
    
    common = [i + 1 for i in range(num_vars) if maskA[i] and maskB[i]]
    onlyA = [i + 1 for i in range(num_vars) if maskA[i]]
    onlyB = [i + 1 for i in range(num_vars) if maskB[i]]
    return common, onlyA, onlyB

def create_split_formula(file_location:str, display_progress:bool = False):
    # === Create hypergraph to partition the formula ===
    _, h_clauses, hyperedges = file_reader.hypergraph_from_cnf_xz(file_location, display_progress)
    queue = Queue()
    p = Process(target=hypergraph_worker.create_partition, args=(queue, h_clauses, hyperedges))
    p.start()
    p.join()
    if p.exitcode != 0:
        if display_progress:
            print(f"Partitioning failed with exit code {p.exitcode} ({file_location})")
    del hyperedges
    

    # === Read the clauses from file ===
    num_vars, num_clauses, clauses = file_reader.read_cnf(file_location)
    result = queue.get()
    partition = result["partition"]


    # === Split clauses is two formula, depending on the partition ===
    # todo: Collect clauses that have only glue variables. Add them to both formulas
    clauses_A = [clause.tolist() for clause, p in zip(clauses, partition) if p == 0]
    clauses_B = [clause.tolist() for clause, p in zip(clauses, partition) if p == 1]
    #clauses_A = [[-21, 37, 40], [-8, 26, 41], [-44, -24, -34], [16, -1, 7], [27, 41, -2], [-47, -20, 27], [7, 48, -8], [14, 48, 9], [-10, 16, 28], [43, 10, 8], [-46, 44, -39], [-36, -28, -12], [-39, -34, -7], [-7, 30, -23], [-22, -2, -27], [-1, -12, -41], [19, -15, -40], [-9, -7, -49], [45, 30, 17], [17, -12, 19], [30, -49, 16], [-2, -40, -11], [43, 3, -37], [44, 3, -41], [34, 10, 48], [1, 46, 20], [39, -41, 9], [11, -24, 14], [-37, 34, 40], [-46, -26, -36], [-45, 37, 19], [11, 44, -8], [-17, 10, 28], [-48, -12, 34], [48, -43, -19], [19, -48, -47], [11, 9, -12], [22, -9, -7], [-15, -1, -8], [7, -11, 17], [-36, 34, 17], [19, 10, 48], [20, -26, -30], [11, -39, -16], [-49, -16, -2], [-8, 48, 10], [-26, 49, -37], [-22, 46, 23], [-19, 24, -47], [28, -49, -20], [-19, 21, 40], [-46, -22, -19], [21, -43, 20], [-41, 1, 26], [-23, -2, 46], [-15, -12, 8], [-28, -24, 26], [-23, 20, -22], [9, -44, 40], [16, 7, 46], [-21, 12, -30], [-48, 22, -7], [-30, 48, 41], [-2, 46, -19], [11, 34, -26], [43, -22, 36], [47, 36, -43], [1, -12, -15], [2, -26, -49], [34, -28, -19], [37, -2, 47], [21, 15, -19], [-19, 46, 30], [30, -22, 39], [46, 1, -16], [43, -30, -22], [-11, 1, -16], [-39, 24, -1], [-27, 40, 10], [-47, 45, -14], [-21, -1, 22], [-3, 34, -28], [10, 9, -24], [-23, 27, -15], [45, 20, -21], [39, -47, -12], [8, 34, 22], [-8, 9, 14], [43, 40, 47], [44, -40, -2], [43, 12, 48], [9, 3, 7], [11, 10, -47], [23, -20, 47], [-37, -34, -9], [-36, 16, -12], [-14, -15, -20], [-30, -21, -16], [1, -2, 40], [44, -45, 19], [-11, -19, -37], [20, -11, -45], [10, -9, 26], [-12, -24, 7], [23, 47, 19], [-9, 39, 30], [3, -36, -2], [28, -8, -20], [26, 47, 49]]
    #clauses_B = [[-4, -44, -34], [21, -35, 41], [-40, 31, -18], [39, -6, -5], [22, -29, 37], [-15, 29, 41], [-42, 32, 6], [-2, -5, -45], [20, 13, -3], [-15, 13, 27], [-33, 37, 30], [-35, 49, 31], [-3, -42, 10], [-3, -18, 49], [42, 31, -18], [-42, -11, -39], [35, 36, 45], [-25, -32, -17], [26, 4, 16], [-25, 35, 21], [42, 27, -9], [-28, -19, 42], [-32, -10, -19], [46, -31, 36], [-40, -36, 42], [9, 6, 45], [38, -2, 26], [33, -30, 19], [-17, -26, -5], [-41, -42, -20], [25, 9, 1], [-11, -49, 29], [6, -31, 38], [50, -37, 10], [36, 39, 25], [38, 34, -36], [-18, -28, 48], [42, 12, 46], [-41, 35, -44], [46, -21, 42], [-31, 28, 34], [24, 33, -18], [37, -42, -29], [-24, -31, -11], [-44, -21, 18], [14, 11, 32], [40, -42, 46], [5, 20, 46], [36, 5, -35], [6, 13, -22], [-32, -33, -41], [36, 29, -20], [2, 35, -28], [-5, 18, -30], [-22, -40, -31], [35, -9, -50], [32, 13, 48], [-48, -16, 32], [25, 33, -11], [-32, 47, -2], [9, -38, 5], [21, 39, -32], [-5, -32, -36], [-24, -48, -38], [17, -37, 31], [38, 15, 9], [29, 42, 5], [-49, 47, -18], [29, 39, -33], [14, -13, -29], [13, -24, 42], [38, 31, -10], [-10, 36, -5], [-38, -18, 24], [33, 29, 48], [30, -25, 36], [29, -34, -21], [16, 18, -27], [14, -50, 49], [4, 10, 28], [-6, 10, 11], [-5, 20, 31], [33, 2, -24], [24, -6, -48], [16, -3, 25], [33, -34, 25], [-5, 10, 27], [27, -39, -33], [-2, 5, -10], [-13, 1, 14], [46, -6, 44], [-17, 48, -35], [-6, 12, -45], [-50, -38, -32], [-1, 50, 36], [-32, 27, 25], [16, 20, 13], [-17, 27, 4], [5, -32, 43], [-42, -32, -11], [35, 38, -37], [-33, -4, -15], [9, 50, -13], [6, 35, 9], [2, -29, 4], [4, 20, 44], [-18, 41, 17], [-41, -13, 3], [39, 25, 6]]
    clauses_all = [clause.tolist() for clause in clauses]
    cnf_A = CNF(from_clauses=clauses_A)
    cnf_B = CNF(from_clauses=clauses_B)
    # Some trailing unused variables may get lost - re-add them to the CNF
    cnf_A.nv = num_vars
    cnf_B.nv = num_vars

    glue_variables, a_variables, b_variables = extract_common_variables(clauses_A, clauses_B, num_vars)
    intersect = set(a_variables).intersection(b_variables)
    assert intersect == set(glue_variables)
    #glue_variables = [1, 2, 3, 9, 10, 11, 12, 14, 15, 16, 17, 19, 20, 21, 22, 24, 26, 27, 28, 30, 34, 36, 37, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49]
    #solution =       [1, -2, 3, 4, -5, 6, -7, -8, 9, 10, -11, -12, 13, 14, -15, 16, -17, -18, -19, 20, -21, 22, 23, -24, 25, -26, 27, 28, 29, 30, 31, -32, 33, 34, 35, 36, -37, 38, -39, -40, 41, -42, 43, -44, -45, 46, -47, -48, 49, 50]
    #limited_sol = [s for s in solution if abs(s) in glue_variables]
    #print(limited_sol)
    #return
    #print(f"Clauses: {num_clauses}, variables: {num_vars}, glue_variables: {len(glue_variables)}, length_clauses: {len(clauses)}")
    #print(f"Glue variables: {glue_variables}")

    # === Solve the partial formulas independently ===
    cnf_whole = CNF(from_clauses=clauses_all)
    start = time.perf_counter()
    with Kissat404(bootstrap_with=cnf_whole) as solver:
        sat1 = solver.solve()
        print(f"Complete solvable: {sat1}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")
        print(f"Solution: {solver.get_model()}")
        #set_stat(file_location, "kissat404", sat1, (time.perf_counter() - start), num_vars, num_clauses)

    start = time.perf_counter()
    partition_solver = PartitionDPLL(num_vars, glue_variables, [clauses_A, clauses_B])

    p_sat = partition_solver.solve()
    print(f"Parition_size: {len(glue_variables)}")
    print(f"Partition Sat: {p_sat}, solved in {1000 * (time.perf_counter() - start):.2f}ms")

    assert p_sat == sat1

    #start = time.perf_counter()
    #division_solver = DivisionDPLL(num_vars, glue_variables, [clauses_A, clauses_B])
    #leared_clauses = [[75, 78, 30, 18, 43, 99, 60, 45], [33, 99, 21, 79, 57], [98, 29, 89, 7, 44, 34, 72, 43], [19, 88, 25, 31], [83, 87, 4, 55, 91, 96, 89, 69], [49, 29, 43, 83, 89, 2, 5], [24, 35, 41, 38, 20, 91, 2, 80], [89, 2, 42, 79, 5, 44, 98, 32, 49], [79, 31, 39, 99, 4, 72, 21, 97], [35, 69, 79, 48, 80, 31, 7, 97, 61, 54], [41, 61, 68, 7, 90, 96, 95, 56, 18], [60, 18, 29, 86, 69, 98, 6], [48, 39, 31, 96], [68, 30, 24, 88, 4, 78, 60, 57], [31, 60, 86, 2, 48, 68], [2, 88, 33, 73, 41, 48, 5], [86, 74, 82, 44, 93, 33, 56, 69, 54], [74, 92, 60, 21, 42, 44, 80, 98, 24], [24, 39, 35, 99, 73, 43, 40, 22, 3], [34, 5, 19, 30, 87, 6, 60], [41, 39, 73, 21, 82, 55, 25], [30, 94, 99, 34, 69, 41, 2, 44, 75, 48, 55, 29, 43, 7, 39, 81, 86, 72, 78, 61], [92, 44, 83, 23, 89, 86, 6, 30, 25, 53, 57, 49], [53, 24, 68, 74, 41, 86, 42, 81, 18, 29, 49], [54, 21, 24, 60, 56, 45, 35, 73, 49, 30, 80, 39], [75, 19, 7, 43, 54], [52, 75, 41, 2, 38, 24, 43, 56, 60, 45], [97, 35, 33, 7, 39, 94, 74, 54], [95, 35, 87, 39, 55, 4, 20, 53, 80], [87, 69, 41, 2, 73, 55, 49, 38, 29, 57, 7, 30, 88, 24, 90, 42, 18, 35], [20, 75, 82, 73, 48, 87, 69, 40, 88, 55, 79], [42, 95, 68, 73, 60, 28, 49, 2], [21, 61, 6, 83], [41, 2, 6, 87, 52, 83], [91, 49, 22, 72, 89, 39], [78, 88, 30, 99, 45, 52, 74, 3, 72, 39], [38, 56, 55, 88, 98, 29, 43, 52], [87, 61, 35, 25, 82, 55, 40], [19, 91, 38, 80], [31, 78, 3, 42, 24, 87, 33, 49, 56, 44, 20], [81, 42, 30, 98, 35, 61, 72, 45, 40], [56, 78, 91, 7, 41, 34, 87, 54], [19, 5, 94], [83, 34, 92, 96, 43], [93, 40, 97, 4, 57, 81, 38, 78, 75, 23, 49], [81, 43, 73, 53, 44], [94, 79, 35, 60, 6, 40, 86, 42, 99, 31], [83, 32, 93, 80, 96, 57, 78, 99], [19, 80, 91], [24, 52, 22, 79, 69, 30, 60, 82, 88], [4, 75, 94, 72, 97, 19, 28, 91, 92, 7, 38, 49], [29, 6, 68, 57, 23, 21, 4, 93, 18, 2, 88, 72], [81, 32, 74, 72, 83, 99, 53, 97], [31, 72, 98, 83, 40, 54, 28], [41, 7, 32, 25, 56], [49, 56, 75, 40, 43, 38], [73, 74, 33, 82, 57, 94, 5], [93, 69, 31, 43, 72, 83, 57, 55, 88], [86, 97, 48, 34, 38, 56, 53, 45], [4, 6, 92, 95, 22, 57, 80, 40], [80, 61, 22, 28, 42, 93, 56, 72, 25], [29, 30, 49, 91, 3, 78, 33, 42, 38, 75, 95, 98, 83, 73, 52, 19, 61, 96, 4, 86], [81, 72, 88, 69, 54, 92], [4, 88, 61, 90, 31, 72], [54, 86, 33, 73, 92, 90, 56, 53, 45, 22], [23, 21, 35, 38, 40, 90, 69, 80, 86, 88], [89, 2, 97, 72, 75, 49], [73, 81, 34, 55, 91, 40, 68], [49, 43, 6, 4, 52, 3, 33, 78], [25, 61, 73, 55, 98, 33, 68], [97, 75, 78, 93, 39, 87, 55, 2, 89, 94], [53, 56, 74, 44, 29, 43, 98, 69], [97, 82, 30, 53, 20, 48, 5, 40], [20, 3, 32, 25], [20, 25, 29, 97, 49], [53, 41, 49, 29, 94, 86, 44, 79, 43, 60, 98, 35, 7], [89, 72, 20, 90, 34, 33, 98, 53, 28, 94, 92], [79, 29, 93, 20, 48, 45, 89, 53], [88, 42, 57, 99, 24, 91, 41, 87, 32, 54], [32, 95, 86, 28, 72, 34], [20, 25, 88, 81, 35, 33, 96], [73, 82, 60, 94, 30, 42, 93, 4, 57], [55, 49, 52, 57], [83, 80, 86, 28, 40, 96, 54], [2, 32, 93, 82, 91, 49, 29, 97, 78, 89], [2, 83, 75, 40], [52, 93, 79, 96, 20], [82, 48, 92, 68, 57, 41, 89, 7], [90, 78, 45, 54, 74, 7, 35, 24, 29, 40], [68, 73, 87, 3, 7, 45, 75, 61, 53], [73, 41, 83, 22, 49, 33, 94, 7, 35, 93], [83, 94, 4, 57, 87, 29, 91, 72], [73, 40, 45, 19, 53], [98, 88, 22, 35, 60, 25], [97, 80, 23, 43], [75, 52, 23, 93, 2, 33], [97, 32, 6, 86, 56, 31, 35], [40, 20, 24, 30, 91, 49], [23, 18, 48, 28, 33, 81, 56, 92], [57, 43, 29, 55, 3, 73, 92, 40], [57, 19, 4, 42, 38, 33, 35, 96], [61, 91, 18, 32, 80], [96, 41, 19, 99, 35, 22, 38, 88, 75], [30, 91, 38, 82, 92, 43, 41, 24, 4, 33, 81, 52, 79, 19, 7, 68, 72, 88], [18, 60, 78, 7, 39, 41, 44, 49, 93, 68], [25, 28, 89, 99, 49], [20, 5, 34, 96], [78, 86, 32, 90, 72, 3], [61, 6, 48, 78, 38, 24, 91, 81, 99, 56], [57, 18, 89, 28, 35, 2, 69, 5, 33], [74, 6, 45, 3, 38, 78, 43], [25, 95, 43, 34], [4, 73, 2, 68, 24, 23, 39], [92, 60, 97, 57, 94, 42, 75, 82], [68, 88, 38, 2, 90, 4, 43], [88, 42, 55, 98, 52, 23, 57, 7, 24, 87, 93, 19, 91], [40, 81, 4, 32, 90, 93, 94, 18, 53, 3], [81, 96, 45, 28, 48, 3, 60], [28, 74, 45, 39, 22, 68], [55, 40, 52, 30, 88, 28, 4, 78, 3, 44, 24, 91], [68, 91, 93, 43, 79, 33, 89, 61], [39, 69, 49, 61, 78, 19, 89], [30, 3, 48, 55, 56, 79], [82, 22, 78, 6, 57, 95, 43, 99, 4, 34, 40, 39], [20, 23, 80, 42, 18, 82, 32, 4], [28, 31, 91, 53, 39, 57, 60, 4, 73], [78, 48, 89, 24, 54, 38, 30, 60, 52, 20], [90, 55, 35, 83, 2, 52], [86, 21, 80, 31, 72, 78, 69, 3], [34, 44, 2, 22, 56, 82, 81, 75, 72, 38, 43, 5, 94], [72, 28, 35, 23, 68, 48, 38, 78], [2, 48, 90, 75, 42, 7, 45, 61, 4, 35, 18], [98, 44, 78, 2, 91, 5, 33, 49], [33, 91, 92, 61, 88, 68, 94, 41, 29], [41, 39, 5, 75, 35, 53, 19], [41, 23, 78, 20, 19, 6, 34, 29, 30, 86], [43, 87, 5, 69, 82, 7, 92, 73, 90], [24, 69, 72, 98, 81, 87, 31, 18, 44, 54], [28, 33, 25, 43, 88], [45, 78, 81, 42, 86], [89, 49, 80, 33, 35, 69], [23, 79, 4, 57, 83, 72, 55, 45, 18, 80], [73, 40, 92, 32], [40, 69, 93, 25, 35, 57, 78, 95], [24, 23, 40, 99, 60, 82, 97, 75, 68], [74, 6, 35, 52, 2, 79, 55, 22, 60, 43, 93, 88], [55, 28, 75, 96, 18], [2, 7, 25, 34, 22, 45], [22, 86, 93, 2, 69, 7, 52, 31, 88, 83], [55, 92, 74, 53, 45, 87, 97, 5], [39, 56, 96], [45, 25, 96, 80, 89, 43], [90, 18, 53, 99, 97, 23], [94, 20, 30, 81, 41, 55, 43, 97, 79, 75], [21, 6, 35, 44, 48, 54, 93, 74, 43, 61], [28, 56, 39, 82, 93, 31, 95, 23, 78, 75], [81, 30, 38, 94, 25, 3, 45, 29, 98, 68], [72, 74, 86, 18, 39, 69, 94, 30], [49, 5, 72, 29], [90, 89, 30, 55, 72, 29, 78], [61, 56, 4, 20, 25], [2, 98, 68, 92, 95, 42, 73, 54], [41, 72, 92, 21, 90], [42, 4, 82, 22, 60, 7, 19, 97, 52], [20, 81, 79, 30, 42, 45, 61, 92, 3, 33, 82, 7], [60, 97, 2, 79, 20, 99], [98, 87, 83, 56, 4, 97, 73, 45, 68, 95, 3, 34], [74, 29, 22, 44, 79, 89, 5], [42, 45, 18, 86, 41, 68, 5], [56, 6, 86, 75], [80, 49, 94, 55, 61, 39], [95, 48, 45, 60, 80, 24, 34], [20, 49, 99], [56, 31, 79, 86, 92, 24, 68], [73, 35, 28, 25, 87, 31], [29, 57, 34, 23, 49, 86], [43, 89, 6, 72, 54, 68, 3], [30, 98, 83, 78, 89, 18], [30, 24, 68, 28, 87, 74, 3, 35, 32, 91], [49, 96, 40, 56, 24], [97, 20, 30, 5, 38, 54, 74], [99, 49, 30, 78, 83, 69, 39], [35, 87, 41, 3, 78, 74, 69, 90, 45, 32, 23, 5], [72, 94, 79, 52, 93, 43, 60], [92, 35, 31, 60, 7, 96, 41, 75], [61, 54, 57, 52, 41, 89], [3, 89, 90, 19, 21, 28, 82, 79, 75, 43, 22], [43, 94, 2, 23, 60, 78, 4, 81, 90, 35, 97, 19], [81, 43, 78, 28, 89], [45, 24, 74, 92, 3, 90, 97, 29, 30, 35, 38, 73, 82, 53, 4, 49, 54], [25, 57, 91, 69, 23, 79, 7, 5], [55, 97, 81, 22, 45, 90, 29, 41, 93, 98], [92, 74, 43, 88, 49, 4, 22], [98, 31, 43, 5], [34, 23, 19, 2, 86], [72, 98, 56, 43, 60, 53, 55, 22, 49, 87, 68], [54, 34, 86, 88, 60, 49, 30, 20], [34, 43, 29, 39, 89, 5], [38, 55, 83, 43, 32, 28], [81, 95, 42, 29, 78, 23, 97], [35, 80, 56, 72, 86, 45], [61, 72, 55, 79, 90, 29, 38, 92], [31, 43, 56, 21, 22, 25], [83, 61, 98, 34, 92, 18, 97], [54, 74, 81, 31, 49, 88, 18, 82, 23, 98, 61, 6], [56, 31, 81, 69, 89, 83, 93, 72], [32, 93, 18, 88, 95, 34, 91, 98, 82, 86, 6], [60, 96, 98, 6, 86, 32, 43, 83], [56, 44, 7, 25, 30, 90, 34], [28, 78, 4, 86, 48, 22], [30, 53, 72, 78, 41, 24, 87, 19, 35, 21, 7, 29], [52, 95, 29, 99, 18, 86, 73, 49, 24, 96], [95, 30, 35, 92, 3, 39], [7, 60, 5, 90, 89, 69, 74, 54, 32, 57, 24], [4, 34, 80, 60, 52, 75, 89], [95, 93, 55, 49, 24, 75, 34, 2, 61, 98, 69, 88, 23], [31, 97, 93, 81, 78, 6, 40, 53], [93, 33, 34, 22, 31, 24, 41, 29], [4, 53, 44, 97, 28, 68, 94, 18], [98, 28, 72, 39, 44, 22, 41], [30, 41, 25, 20, 39, 91, 42], [42, 82, 52, 32, 72, 29, 45, 99], [87, 88, 55, 45, 18, 93, 96], [88, 40, 3, 61, 98, 86, 39, 53], [34, 80, 25], [18, 69, 32, 30, 89, 56, 28, 72], [92, 21, 44, 29, 43, 18, 82, 75, 91], [93, 73, 22, 32, 52, 31, 94, 88, 87, 80, 90, 49], [42, 61, 41, 57, 79, 44, 2, 83, 22, 81], [48, 34, 89, 72, 43], [34, 53, 99, 88], [33, 79, 83, 60, 43, 88, 74, 73, 80], [33, 89, 35, 99, 61, 42, 48, 25, 96], [93, 78, 54, 31, 49, 95, 98, 87, 2, 19], [88, 90, 5, 52, 29], [82, 96, 89, 42, 92, 33, 40, 74, 69, 6], [68, 96, 82, 42, 56, 40, 30, 79, 4, 18], [99, 42, 20, 38, 7, 86, 91, 92], [69, 42, 20, 89, 40, 49], [34, 18, 61, 20, 87, 31, 94, 72], [49, 97, 29, 6, 93, 35, 20, 55, 99], [61, 54, 41, 99, 7, 82, 87, 34, 42, 93, 39], [43, 80, 53], [35, 61, 72, 82, 21, 6, 31, 81, 96], [34, 53, 72, 61, 22, 57, 83, 7, 93, 86, 43, 30, 24], [94, 24, 3, 88, 38, 96, 99, 87, 92, 52, 23], [28, 3, 30, 33, 5, 44, 52, 43], [92, 5, 94], [3, 57, 96, 89, 99, 41, 94, 34, 79], [43, 24, 93, 72, 35, 96, 90], [56, 89, 72, 48, 60, 75, 45], [30, 25, 87, 34], [80, 88, 28, 53, 96, 86], [89, 18, 22, 72, 92, 40], [43, 81, 68, 99], [29, 53, 74, 45, 87, 38, 61, 56, 54, 69, 43], [48, 44, 97, 22, 5, 20, 89, 38, 61, 18, 90], [45, 89, 3, 7, 74, 22, 40], [42, 39, 2, 75, 97, 54, 86, 94, 49, 56, 19], [48, 31, 18, 42, 82, 39], [55, 53, 24, 86, 18, 73, 98, 43, 4, 28], [79, 48, 31, 23, 88, 34], [91, 87, 72, 43, 22, 38, 30, 89, 82, 29, 34], [18, 6, 97, 98, 60], [38, 75, 98, 86, 43, 18, 92], [35, 18, 93, 3, 68, 61, 30, 83], [29, 44, 40, 60, 30, 23, 75, 80], [38, 55, 45, 40, 49, 24], [53, 82, 80, 40, 60, 7, 19, 32, 88], [79, 57, 19, 25, 39, 89, 73], [2, 33, 93, 83, 44, 75], [48, 86, 73, 82, 18, 96, 43], [34, 57, 25, 41, 6, 18], [80, 72, 35, 53, 75], [43, 90, 55, 19, 29, 53, 81, 48, 99], [22, 89, 21, 38, 96, 43, 5, 49], [18, 87, 32, 21, 55, 38, 93, 90, 69, 95], [30, 32, 48, 34, 53, 29, 60, 2, 56, 44, 78, 69, 72], [82, 38, 88, 75, 81, 92, 23, 28, 72, 91], [68, 56, 89, 72, 3, 75, 40], [69, 52, 24, 38, 82, 96, 55, 98, 48, 87, 90, 2, 29], [83, 35, 5, 93, 30, 60, 41, 79, 39], [94, 81, 48, 97, 72, 88, 44, 32, 98, 31], [28, 45, 23, 81, 56, 5], [73, 24, 49, 39, 30, 89], [33, 49, 60, 4, 89, 94, 54, 82, 56, 74], [5, 80, 38, 19, 49, 75], [54, 86, 88, 24, 38, 23, 3, 49, 57, 41, 93, 44, 6, 42], [57, 78, 44, 25, 73], [72, 29, 97, 30, 86, 2, 21, 91, 32, 92], [3, 95, 29, 56, 7, 45, 61, 88], [55, 3, 95, 45, 68, 74, 49, 80, 25, 88], [3, 21, 28, 93, 23, 89, 24, 56, 35, 52, 80, 38, 45, 99], [73, 42, 21, 49, 83, 57, 81, 19, 31, 74, 86, 34], [98, 40, 93, 44, 87, 4, 82, 89, 60, 81, 56, 30, 90], [54, 38, 4, 68, 95, 45, 32, 60, 20], [75, 97, 39, 94, 2, 93, 40, 91, 81, 57, 21, 88, 6], [3, 79, 60, 91, 88], [81, 38, 53, 5, 75, 48, 72, 19], [87, 61, 94, 72], [20, 52, 44, 74, 79, 83], [96, 82, 73, 43, 75, 54, 94, 39], [41, 39, 22, 42, 94, 24, 98, 49], [40, 32, 23, 48, 73, 97, 19], [98, 56, 75, 80, 31, 92], [82, 97, 92, 53, 43, 48], [38, 28, 94, 80, 30, 75, 3, 33, 7, 83], [61, 79, 29, 21, 99, 18, 43, 34], [5, 79, 56, 20], [88, 81, 48, 93, 34, 2, 21, 24, 53, 7, 91, 43, 40, 57, 5], [31, 57, 48, 7, 3, 53, 38, 60, 5], [48, 3, 31, 86, 91, 73, 44, 88, 60, 22, 41, 6], [81, 32, 91, 61, 97, 21, 41, 22, 92, 78, 3, 45], [54, 38, 28, 72, 24, 98, 49, 94, 68], [52, 21, 81, 19, 68, 7, 4, 99, 60, 78], [2, 97, 57, 88, 5], [19, 54, 80, 2, 68, 92], [5, 29, 7, 95, 19], [5, 99, 69, 39, 87, 29], [68, 86, 6, 3, 57, 28, 83, 39], [83, 2, 86, 40, 30, 29, 20, 89, 45], [33, 97, 95, 69, 43, 29, 2, 4, 21, 93, 60, 45], [61, 95, 87, 7, 5, 31, 72, 55, 25], [83, 32, 52, 56], [34, 39, 89, 55, 57, 18, 45, 30], [25, 33, 41, 38, 79, 22], [18, 32, 22, 87, 4, 53, 56, 61, 20, 99], [90, 32, 78, 35, 18, 94, 44, 7, 48], [3, 56, 82, 28, 49, 44, 68, 4, 61], [22, 97, 93, 28, 4, 79, 89, 56, 35], [6, 48, 75, 99, 61, 43], [80, 3, 30, 45, 48, 29, 25, 39], [90, 6, 41, 72, 18, 25, 52], [20, 69, 92, 45, 90, 81, 7], [7, 21, 80, 54, 18, 69, 43, 78, 2, 89], [87, 24, 61, 22, 6, 72, 95, 98, 91, 93, 18], [2, 82, 53, 80, 72, 48, 42, 23, 75], [3, 38, 32, 49, 28, 19, 20, 94, 98, 25], [95, 68, 86, 41, 99], [82, 45, 72, 89, 30, 78], [2, 82, 30, 29, 78, 34, 93, 5, 81, 88], [53, 4, 49, 39, 18, 57, 75], [73, 80, 35, 44, 48, 6, 99, 89, 57, 39], [79, 39, 45, 28, 25], [83, 88, 28, 95, 61, 3, 69, 48, 56, 42, 38], [32, 88, 87, 31, 20, 80], [2, 49, 24, 73, 78, 33, 41, 60, 20], [54, 3, 90, 22, 52, 19, 4, 92, 94, 83, 57, 75, 6], [24, 7, 94, 23, 72, 18, 35], [52, 81, 83, 94, 38, 86, 33, 30, 35, 54, 25], [34, 90, 31, 19, 7, 28, 96, 94, 40], [45, 57, 97, 69, 88, 40], [53, 91, 39, 18, 57, 29, 7, 5], [96, 49, 73, 31, 39], [24, 82, 22, 96, 39], [94, 73, 42, 68, 78, 18, 34], [38, 79, 74, 92, 73, 87, 61, 7, 5, 22], [6, 29, 54, 86, 94, 21, 98, 56, 38, 43, 19], [95, 99, 25, 91], [95, 39, 34, 73, 68], [33, 43, 52, 19, 80], [68, 83, 61, 25, 79, 40], [18, 97, 5, 42, 94, 90, 57, 20, 23], [57, 78, 29, 92, 96, 75, 21, 55, 7, 49], [78, 31, 82, 3, 88, 22, 93, 25], [82, 90, 72, 78], [68, 96, 83, 49, 75, 33, 21, 55, 4, 94], [38, 5, 95, 32, 86, 56], [45, 54, 60, 24, 57, 79, 19, 96, 4, 40, 43, 88], [93, 41, 82, 6, 80, 21, 86, 60, 57, 31], [61, 74, 28, 23, 31, 34, 91, 94, 89, 33, 86], [69, 90, 40, 55, 34, 75, 45], [35, 91, 86, 82, 33, 61, 25, 38, 98], [45, 60, 96, 68, 88, 92, 79, 43, 39], [31, 72, 86, 42, 45], [94, 22, 28, 89, 96, 56, 19, 40], [24, 69, 87, 30, 60, 91, 40, 72, 5], [91, 4, 55, 97, 42, 30, 98], [83, 55, 93, 30, 52, 61, 43], [30, 74, 6, 44, 82, 2, 68, 56], [88, 34, 80, 25], [55, 88, 23, 18, 41, 28, 57, 90, 82, 74, 86], [7, 53, 20, 60, 25, 87], [19, 44, 43, 3], [81, 54, 7, 29, 44, 40, 91, 78, 92], [40, 43, 18, 74, 83, 99], [93, 60, 73, 38, 57, 68, 6, 79, 55, 29, 99], [57, 45, 33, 6, 25, 90, 99], [73, 82, 79, 48, 44, 53], [7, 4, 75, 52, 93, 55, 44, 82, 68], [40, 87, 2, 96, 61], [33, 96, 68, 38, 87, 21, 34, 55, 24, 31, 42, 79, 81], [53, 88, 6, 49, 96, 45, 29, 34], [45, 93, 55, 5], [75, 88, 45, 91, 83], [49, 79, 23, 2, 53, 19, 80, 32], [60, 91, 87, 98, 72, 34, 97, 3], [44, 82, 6, 80, 73, 41, 23, 28, 5], [79, 30, 29, 35, 21, 52, 54, 7, 4, 44, 61, 49], [18, 92, 91, 21, 83, 52, 48], [25, 18, 88, 44, 80, 20], [73, 23, 43, 49, 7, 38, 4, 40], [28, 57, 90, 82, 49, 35, 44, 78, 22], [86, 90, 39, 3, 75, 95], [38, 45, 22, 5], [79, 97, 91, 44, 7, 40, 56, 2, 42, 98, 83, 61], [24, 29, 91, 74, 78, 73, 4, 19, 60], [69, 61, 30, 52, 35, 94, 56, 73, 88, 92, 7, 39], [41, 7, 30, 96, 74, 22, 35, 39], [44, 33, 21, 60, 69, 25, 18, 7, 41, 39], [19, 79, 7, 60, 20, 83, 73, 33, 40, 69, 54], [99, 38, 29, 21, 68, 87, 5], [31, 48, 3, 6, 74, 43], [88, 40, 21, 60, 74, 42, 22, 91, 44, 38], [79, 4, 97, 94, 41, 25, 83, 86, 23, 93, 49], [91, 24, 38, 22, 49, 89, 3, 20], [21, 3, 38, 31, 35, 56, 78, 69, 44, 87, 55, 72], [86, 6, 49, 79, 61, 98, 41, 34], [19, 35, 98, 55, 60, 33, 87, 4, 81, 74, 52, 29], [44, 80, 82, 61, 54, 74, 98, 73, 78, 38], [49, 54, 57, 89, 18, 32, 20], [20, 69, 87, 44, 33, 75, 54, 79], [79, 80, 32, 44, 52, 75], [88, 28, 43, 21, 44, 68], [48, 28, 75, 43, 6, 72, 94], [98, 41, 4, 25, 39, 73], [87, 44, 42, 93, 23, 99, 40], [48, 90, 38, 22, 61, 45, 79, 74, 21, 33], [44, 41, 23, 53, 97, 82, 28, 92, 98, 4, 2], [41, 24, 44, 91, 53, 60, 79, 68], [97, 99, 68, 56, 42, 6, 32], [35, 99, 32, 30, 55, 56], [68, 60, 49, 33, 39], [69, 28, 91, 5, 45], [68, 23, 75, 86, 25, 20], [38, 75, 24, 44, 20, 52, 90, 56], [24, 22, 42, 49, 41, 52, 90], [6, 69, 52, 74, 40, 81, 96, 43], [81, 86, 96, 79, 49, 21, 3, 7, 31], [68, 3, 25, 31], [69, 53, 5, 45], [56, 41, 75, 42, 60, 31], [18, 78, 48, 45, 95], [22, 73, 92, 56, 45, 87, 29, 42, 74, 95], [23, 99, 81, 35, 32, 30, 82, 89, 68, 53], [30, 69, 95, 18, 4, 41, 2, 78, 57, 99, 33, 23], [45, 92, 42, 79, 96, 60, 3, 18, 69, 95], [43, 44, 54, 6, 78, 72, 29, 81, 5], [86, 4, 54, 21, 45, 25, 96, 18], [57, 4, 93, 90, 41, 39, 83], [57, 21, 52, 22, 35, 79, 61, 73, 81, 39, 40, 69, 98], [60, 78, 80, 72], [22, 45, 31, 89, 3, 99, 86], [93, 6, 86, 38, 28, 23, 79, 72, 44, 57, 4, 99, 97, 80], [55, 32, 57, 69, 42, 95, 38, 72, 87, 74, 89, 41], [42, 92, 45, 25, 39], [22, 3, 91, 39, 97, 19, 75], [49, 91, 18, 20], [72, 60, 35, 55, 45, 83, 19, 86], [48, 22, 31, 60, 97, 78, 21, 38, 68], [42, 31, 39, 99, 94, 78, 18, 81, 97], [91, 95, 83, 28, 68, 41, 57, 93, 87, 38, 99, 33, 55, 73, 30, 43], [94, 69, 18, 61, 7, 93, 52, 44, 5], [74, 81, 34, 39, 25, 83, 6, 23], [98, 80, 22, 57, 69, 87, 34, 97, 21, 7, 25], [30, 99, 3, 5], [96, 29, 74, 41, 3, 90, 45, 98, 25], [2, 22, 55, 69, 19, 21, 87, 79, 83, 96, 91], [78, 18, 96, 34, 38, 22, 81, 60, 49], [39, 32, 92, 52, 5], [99, 69, 32, 72, 52, 89], [95, 81, 73, 69, 89, 52, 35, 61, 54], [94, 40, 87, 54, 33, 99, 91, 88, 5], [81, 94, 28, 45, 30, 34, 69, 43], [98, 38, 40, 19, 48, 83, 55, 57, 91], [91, 93, 68, 19, 28, 54, 43, 33], [28, 49, 91, 22], [73, 93, 41, 44, 35, 30, 79, 89, 20, 74, 86], [93, 18, 74, 31, 20, 44, 33, 35], [38, 25, 54, 21, 29, 72, 41, 32, 52, 49], [30, 89, 82, 68, 39], [18, 35, 96, 60, 94, 81, 32, 55, 87], [86, 7, 54, 20, 38, 3, 97, 53, 33], [20, 60, 40, 42, 18, 69, 45], [41, 23, 54, 28, 98, 90], [28, 60, 75, 20, 5], [38, 3, 60, 86, 6, 49, 68, 81, 55, 72, 30, 57, 20, 28], [98, 95, 96, 88, 30, 80, 42, 72, 45, 87, 4, 3, 18], [2, 21, 32, 45, 30, 39], [24, 18, 79, 73, 43, 34, 40, 28], [81, 91, 29, 69, 32, 40, 72, 30, 61, 92], [49, 56, 90, 45, 24, 20, 2, 61, 89, 52], [96, 80, 45, 68, 25, 42, 2, 94], [20, 83, 93, 33, 52], [73, 33, 92, 3, 28, 39, 40, 80], [61, 31, 79, 55, 52, 40, 82, 43], [80, 29, 18, 99, 40, 54, 57, 45], [25, 45, 40, 3, 86, 7, 22], [38, 81, 23, 72, 43, 90, 34], [6, 80, 52, 97, 92, 5], [21, 88, 31, 42, 3, 87, 24, 95, 39, 72], [6, 86, 73, 74, 55, 34, 43], [49, 23, 6, 80, 79, 86, 82, 69], [32, 82, 57, 23, 75, 30, 48, 6, 54, 79, 34], [82, 43, 41, 48, 69, 54, 86, 56, 99], [88, 45, 90, 69, 28, 96, 94, 43, 2], [45, 91, 56, 4, 7, 55, 86, 61], [81, 99, 29, 91, 95, 61, 18, 93, 68], [41, 88, 61, 72, 98, 25, 57, 29, 87], [38, 52, 95, 88, 90, 41, 97], [45, 79, 93, 81, 60, 42, 96, 35, 22, 75, 56], [83, 69, 38, 19, 29, 79, 40, 21, 96, 86, 49], [93, 94, 75, 81, 18, 99, 60, 38, 83, 5], [7, 5, 52, 91, 95, 97], [25, 32, 60, 48, 98, 39, 72, 54, 87], [92, 80, 94, 2, 75, 69, 42, 83], [25, 69, 55, 4, 53, 31], [33, 75, 81, 99, 6, 19, 24, 40], [80, 7, 25, 21, 44, 22, 87, 96, 34], [19, 44, 96, 99, 39], [72, 22, 33, 55, 44, 25, 78, 69, 3, 18], [60, 52, 83, 97, 88, 91], [25, 31], [3, 31, 6, 18, 29], [34, 43, 29, 2, 19, 32, 91, 94, 52, 83], [73, 79, 33, 48], [82, 6, 49, 18, 96, 40], [29, 22, 20, 78, 75, 3, 52, 44, 93, 69, 61, 97, 95], [56, 91, 54, 31, 53, 42, 87], [99, 94, 32, 73, 55], [23, 3, 52, 81, 20, 61, 57, 31, 24, 43], [99, 48, 39, 86, 88, 79], [90, 44, 31, 98, 75], [56, 31, 97, 75, 18, 81, 21, 94, 33], [20, 55, 7, 93, 75, 31, 94, 83], [86, 3, 55, 41, 75, 45, 48], [82, 6, 2, 91, 31, 69, 61, 56, 55, 87, 89], [92, 25, 89, 86, 22, 18], [7, 23, 93, 48, 68, 89, 79], [78, 41, 94, 83, 4, 20, 44, 7, 3, 48, 81], [35, 75, 5, 96, 39], [83, 93, 38, 86, 42, 3, 23, 68, 49, 44, 99, 53], [94, 61, 24, 7, 86, 5, 72, 35, 18], [94, 93, 6, 98, 2, 57, 69, 35, 61, 28, 45], [25, 74, 43, 83, 73], [91, 32, 43, 39, 89, 4, 96], [87, 28, 61, 93, 88, 25], [97, 22, 7, 34, 3, 68, 4, 74, 60], [87, 81, 98, 60, 83, 78], [21, 93, 74, 23, 32, 30, 96, 69, 90, 35, 43, 56], [34, 45, 81, 54, 42, 41], [41, 34, 93, 3, 6, 5, 30, 53, 78, 82], [79, 25, 80, 39], [33, 2, 75, 43, 35, 92], [5, 88, 81], [52, 96, 94, 56, 43, 89, 74], [56, 88, 99, 73, 23, 74, 48, 34], [88, 45, 75, 78, 7, 49, 96, 40], [31, 56, 39, 75, 80, 42], [95, 80, 35, 61, 48, 86, 78], [6, 74, 3, 78, 90, 5, 31, 24, 40], [4, 97, 42, 30, 86, 57, 23, 78, 28, 25, 6, 98, 49], [92, 68, 6, 54, 57, 90, 60, 78], [34, 38, 88, 78, 68, 4, 7, 73], [18, 94, 41, 93, 4, 24, 91, 49, 45], [29, 89, 19, 39, 90, 49], [38, 35, 54, 72, 89, 91, 23, 21, 53], [98, 95, 18, 39, 48], [30, 28, 3, 61, 98, 89, 19, 24, 23, 80, 39], [24, 56, 92, 23, 60, 88], [91, 42, 99, 38, 74], [30, 23, 44, 43, 94, 28, 56], [38, 97, 94, 31, 78, 81], [30, 45, 38, 32, 92, 23, 2, 99, 6, 96], [78, 31, 95, 69, 34, 55, 2, 5], [25, 99, 7, 45], [2, 81, 40, 96, 86, 43, 38], [82, 60, 88, 21, 72, 99, 2, 22, 93, 74, 97, 32], [92, 38, 88, 33, 54, 7, 69, 44, 74], [87, 7, 60, 78, 68], [6, 54, 22, 86, 52, 30, 82, 92, 89, 75], [57, 60, 33, 41, 88, 98, 25, 91], [88, 44, 2, 32, 31, 99, 72, 61, 68], [72, 86, 19, 7, 34, 55, 32, 78, 52, 93, 68], [39, 78, 30, 29, 45], [18, 79, 32, 68, 73, 48, 74, 34], [98, 19, 61, 52, 82, 54, 21, 31, 2, 74], [25, 23, 34], [4, 82, 90, 93, 20, 56], [22, 29, 89, 81, 32, 72, 87, 60, 78], [79, 53, 72, 82, 89, 43], [80, 78, 54, 42, 20, 92, 91], [73, 61, 56, 25, 93, 6, 32], [30, 39, 23, 18, 55, 32], [38, 2, 81, 5, 6, 28, 97], [75, 35, 88, 30, 93, 99, 95, 21, 49, 52, 68], [28, 61, 43, 54, 81, 24], [33, 3, 42, 23, 88, 87, 41, 57, 94, 72], [94, 81, 45, 43, 56, 93], [25, 92, 82, 97, 45, 40], [80, 98, 35, 28, 25, 33, 56, 78, 38, 48, 75], [19, 56, 61, 20, 55, 39, 2, 23, 45, 48, 72], [57, 49, 23, 38, 98, 34, 82, 90, 93], [49, 35, 25, 68, 61, 53, 98, 2, 74, 88], [40, 30, 82, 78, 2, 49, 19, 39], [3, 45, 99, 78, 87, 82, 61, 29, 35, 72], [7, 75, 24, 93, 91, 52, 55, 31, 87, 96], [98, 97, 19, 6, 61, 29, 3, 88, 94, 53, 38, 79, 35, 55, 75], [60, 30, 23, 45, 4, 29, 2, 35, 54], [69, 52, 56, 93, 75, 6, 34, 87, 31], [54, 30, 3, 40, 18, 43], [48, 31, 91, 28, 41, 94, 57, 38], [89, 28, 55, 78, 4, 2, 98, 30, 41, 94, 92], [78, 86, 53, 89, 32, 19, 69, 20, 24, 6, 3], [90, 80, 42, 24, 44, 18, 20, 98, 87, 82, 30, 74], [75, 23, 43, 21, 52, 45, 32, 96, 28], [82, 81, 52, 6, 19, 56, 49, 61, 93, 97, 43], [95, 40, 56, 28, 81, 87], [79, 69, 53, 18, 32, 34, 39, 24, 57, 55, 23], [18, 78, 2, 41, 92, 35, 31], [72, 3, 81, 68, 5], [97, 98, 5], [49, 53, 33, 6, 98, 89, 92, 54, 4, 25], [81, 42, 53, 54, 72, 95, 3], [4, 52, 79, 69, 86, 39, 91, 60], [24, 29, 45, 40, 22, 2, 72, 78], [35, 24, 88, 55, 22, 98, 83, 94, 90, 53, 75], [39, 40, 88, 74, 4, 53, 43, 2, 91, 29, 60, 93, 83], [74, 82, 96, 35, 5, 40, 30, 93, 22, 86, 42, 55, 81, 79], [87, 44, 52, 28, 68, 19, 55, 88], [33, 52, 31, 89, 86, 25], [23, 95, 32, 42, 92, 56, 6, 61, 74], [69, 5, 38, 45], [49, 35, 42, 3, 87, 40], [83, 99, 7, 39, 42, 87, 88, 34, 23], [52, 48, 54, 98, 94], [45, 83, 61, 49, 38, 33, 94, 98, 54, 93], [98, 94, 92, 20, 89, 74, 97, 41, 80, 68, 4, 34], [28, 48, 68, 4, 45, 88, 73, 22, 91], [57, 82, 80, 86, 42, 35, 44, 40, 60, 19, 3], [88, 90, 49, 57, 32, 18, 29, 24, 34, 55], [93, 22, 78, 57, 4, 91, 98, 2, 33, 86, 40, 96], [60, 30, 78, 3, 91, 45], [48, 31, 68, 45, 54], [43, 80, 6], [35, 86, 25, 83, 56, 32], [18, 98, 92, 32, 21, 42, 89, 60, 82, 52, 87, 25], [6, 42, 74, 54, 25, 23, 73, 2], [98, 54, 29, 6, 39, 4, 91, 34, 32, 52], [35, 21, 2, 41, 81, 39, 25, 18, 89, 94, 92], [49, 22, 54, 61, 69, 78, 53, 72, 39], [33, 81, 23, 48, 6, 93, 42, 39, 19, 97, 52, 2], [32, 78, 23, 24, 80, 74, 88, 61, 6, 5], [96, 20], [32, 52, 35, 68, 80, 24, 83, 38, 93, 48, 42, 94], [31, 93, 99, 24, 73, 44, 56, 22, 3, 86, 89, 75, 69, 82, 6], [40, 7, 52, 87, 83], [90, 55, 61, 43, 74, 3], [20, 96], [57, 32, 4, 87, 21, 74, 93, 22, 35, 19, 96], [28, 83, 73, 78, 75, 34], [89, 3, 92, 86, 34, 74, 33, 48, 69, 83, 40, 94], [7, 25, 57, 75, 52, 95, 33], [44, 5, 38, 91, 55, 97, 92], [2, 81, 72, 97, 94, 83, 60, 99], [29, 60, 35, 30, 93, 5, 80], [86, 99, 45], [35, 89, 29, 33, 87, 92, 45, 2], [57, 73, 99, 96, 43, 60, 75, 32], [98, 4, 83, 18, 74, 54, 52, 48], [82, 69, 3, 18, 40, 60, 7, 43], [91, 44, 55, 48, 82, 94, 20, 86, 75, 52], [56, 68, 6, 44, 96], [96, 35, 87, 2, 69, 33, 73, 19], [34, 23, 97, 56, 33, 60], [99, 80, 88, 5], [19, 94, 89, 80, 99, 25], [35, 69, 33, 73, 98, 92, 28, 45, 75], [35, 92, 72, 19, 5, 52, 94], [38, 18, 55, 93, 74, 68, 45, 48, 22, 43], [73, 69, 25, 43], [79, 75, 33, 73, 61, 69, 54, 99, 30, 80], [88, 19, 83, 5, 24, 52, 92], [19, 99, 89, 29, 2, 42, 91, 30, 41, 74], [93, 81, 86, 44, 95, 34, 72, 75, 99, 78, 96], [43, 54, 80, 61], [23, 35, 83, 7, 81, 94, 73, 41], [98, 72, 45, 69, 20, 79, 57], [99, 24, 94, 19, 89, 32, 79, 49], [52, 41, 48, 5, 38, 20, 56], [86, 40, 94, 24, 56, 83, 69, 7, 5], [99, 90, 33, 44, 23, 96, 60, 34], [34, 83, 45, 21, 54, 56, 90, 49, 25], [4, 94, 72, 74, 43, 2, 18, 38], [38, 86, 78, 94, 97, 80], [4, 61, 55, 48, 28, 73, 39, 22, 56, 21], [93, 69, 95, 32, 60, 19, 96, 88], [39, 82, 28, 55, 30, 56, 79, 33], [30, 48, 73, 23, 99, 6, 74, 91], [49, 2, 18, 86, 31, 20], [98, 19, 32, 78, 45, 29, 57, 73, 34, 55], [21, 49, 78, 83, 31, 93, 69, 57, 25], [28, 94, 87, 34, 96, 43], [3, 42, 20, 41, 35, 7, 24, 29, 45, 96], [25, 75, 44, 39], [57, 6, 48, 74, 39, 97, 28, 35, 94, 55, 31, 33, 40, 80], [91, 57, 20, 82, 80, 43, 94, 44, 60, 25], [98, 52, 86, 43, 32, 24], [38, 33, 69, 23, 98, 28, 82, 79, 97, 7, 93], [79, 91, 32, 72, 25, 68, 28, 93, 4, 94], [78, 83, 69, 45, 56, 99], [94, 38, 6, 44, 87, 98, 53, 40], [20, 82, 35, 18, 88, 52, 61, 39, 29], [7, 94, 35, 89, 33, 25, 75, 87], [94, 28, 2, 44, 33, 7, 87, 20, 61, 23], [94, 32, 5, 22, 68], [24, 44, 96, 99, 75, 49], [81, 89, 60, 56, 42, 90, 74, 30, 3, 22, 4, 93, 35, 83], [96, 38, 18, 42, 22, 74, 31], [38, 23, 97, 88, 55, 94, 34, 79], [98, 79, 95, 20, 89, 18], [68, 4, 97, 54, 21, 57, 22, 19, 28, 24, 90, 45, 82, 3, 41, 53], [23, 25, 75, 72, 55, 52, 96, 40, 49], [94, 91, 28, 53, 69, 55, 6, 81, 3, 61, 93, 88, 87], [82, 79, 68, 92, 22, 87, 4, 18, 41, 34], [88, 86, 40, 42, 55, 96, 74, 72, 82, 5], [97, 80, 72, 78, 38], [48, 79, 56, 38], [19, 2, 57, 21, 90, 98, 52, 35, 29], [32, 39, 40, 81, 2, 95, 97, 75], [92, 86, 30, 38, 19, 49, 42, 95], [57, 29, 72, 4, 7, 93, 78], [94, 18, 33, 79, 88, 44, 61, 90, 82, 49, 7], [30, 92, 97, 18, 40, 29, 53, 6], [98, 48, 29, 57, 40, 44, 54, 96, 31], [97, 93, 55, 21, 19, 94, 25, 68], [19, 33, 68, 72, 20, 24, 75], [45, 2, 69, 74, 48, 87, 34, 61, 79, 40], [6, 19, 43, 21, 96, 92, 40], [80, 96, 74, 89, 69, 29, 61, 94, 49], [90, 93, 25, 80, 7, 30, 72, 75], [23, 43, 20, 48, 30, 39, 45], [95, 29, 32, 81, 25, 54, 3], [90, 25, 96, 86, 56, 7, 68, 72], [31, 99, 25], [55, 43, 44, 2, 23, 94, 83, 73], [34, 60, 53, 54, 44, 75], [31, 95, 5, 61, 39], [28, 43, 54, 33, 38, 48, 53, 21, 6], [28, 45, 92, 19, 3, 4, 32, 31], [48, 72, 20, 5, 97, 22, 89, 99, 24, 53, 61, 55, 38, 34, 82, 92], [80, 68, 28, 61, 7, 78, 52, 35, 22, 45, 20], [60, 81, 78, 29, 25, 7, 86, 4, 48, 30, 35, 41, 99, 53], [86, 39, 68], [43, 25, 74, 79, 81, 95, 35, 57, 61, 44, 93, 91], [73, 39, 98, 32, 3, 97, 74, 94, 88, 31, 35, 18, 82, 53], [86, 43, 4, 81, 35, 52, 89], [98, 78, 93, 89, 53, 31, 5, 81], [49, 72, 91, 7, 32, 35, 57], [74, 24, 35, 40, 83, 99, 20], [57, 78, 95, 96, 7, 60, 38, 18, 98, 92, 44, 43], [88, 33, 52, 80, 23, 78, 60, 75, 99, 19], [74, 45, 28, 69, 23, 97], [92, 48, 53, 21, 89, 83, 45, 99], [20, 60, 5], [32, 87, 6, 56, 20, 43], [72, 82, 29, 32, 42, 75, 19, 90], [92, 86, 98, 83, 28, 78, 80], [69, 44, 99, 90, 74, 79, 61, 54, 86, 41, 7, 5], [4, 28, 53, 30, 82, 90, 60, 97, 32, 54, 94, 39, 23, 98], [52, 86, 28, 41, 89, 56, 49, 20], [45, 82, 98, 79, 39, 72, 80], [88, 24, 30, 60, 53, 44, 32, 78, 20, 69, 75], [18, 82, 75, 35, 79, 56, 25, 43, 96, 91, 93, 54], [23, 29, 2, 96, 75, 72, 57, 69, 48, 42, 61], [55, 2, 88, 79, 38, 87, 23, 48, 29, 19, 93], [96, 30, 52, 20], [74, 95, 83, 99, 68, 39], [5, 61, 41, 95, 91, 2, 24, 19], [87, 83, 21, 56, 98, 61, 19, 42, 49, 34, 33, 6], [53, 49, 43, 91, 22, 18, 25], [49, 19, 7, 23, 28, 56, 94, 86, 91], [20, 31, 45, 96], [91, 40, 72, 61, 54, 56, 38, 5], [96, 99, 45, 19, 31, 55], [34, 24, 99, 19, 79, 87, 4, 69, 31, 3], [89, 30, 94, 38, 6, 57, 72, 90, 55, 43], [69, 96, 31, 3, 88, 55, 52, 4, 80, 86], [48, 34, 68, 18, 73], [7, 83, 25, 30, 98, 92], [73, 90, 68, 23, 83, 20, 45, 54], [45, 4, 24, 18, 30, 38, 54, 33, 99, 48, 3, 43], [29, 94, 75, 90, 55, 60, 42, 6], [20, 90, 87, 52, 2, 38, 60, 5], [86, 20, 90, 35, 3, 94, 89, 18], [52, 82, 60, 5, 25, 56], [7, 74, 44, 55, 30, 98, 78, 88, 48, 2, 97, 28, 20], [80, 28, 23, 4, 32, 38, 34], [54, 93, 81, 74, 40, 4, 7, 52, 56, 69, 97, 91, 2, 33, 45], [72, 43, 5, 21], [97, 32, 79, 95, 3, 31, 80], [30, 87, 28, 96, 49, 72, 52, 79, 4, 56, 91], [25, 18, 54, 5], [91, 96, 24, 28, 20], [49, 56, 22, 35, 33, 6, 52, 90, 86, 29, 95], [2, 55, 49, 6, 89, 45, 42, 99], [97, 31, 90, 29, 56, 55, 21, 92, 98], [83, 97, 93, 61, 3, 87, 75, 49], [79, 74, 73, 32, 34, 87, 25], [91, 89, 53, 82, 80], [72, 48, 78, 68, 97, 55, 81, 32], [53, 56, 86, 97, 94, 33, 73], [38, 42, 21, 86, 60, 79, 95, 44, 32, 24], [34, 38, 25], [5, 21, 80, 74, 41, 68], [25, 99, 87, 7, 5, 93, 49], [95, 5, 73, 83, 92, 48, 42, 7, 87, 55, 57, 80, 28, 97], [7, 80, 49, 54, 32, 87, 38, 22, 20], [72, 2, 25, 42, 82, 45, 5], [6, 75, 81, 92, 49, 95, 40], [79, 35, 39, 55, 68, 20], [97, 18, 39, 22, 41, 5, 95], [45, 39, 98, 42, 28, 82, 53, 79, 20, 3, 19, 5], [7, 34, 55, 19, 92, 82, 5], [31, 3, 23, 78, 43], [30, 6, 42, 88, 38, 78, 18, 2, 90], [53, 86, 73, 57, 7, 29, 43], [68, 25, 99], [96, 48, 35, 81, 82, 87, 21, 54, 41, 93, 94, 6], [7, 81, 31, 88, 21, 94, 92, 75, 32, 60], [39, 86, 69, 79, 22, 82, 90, 34, 49], [95, 96, 61, 86, 34], [94, 25, 92, 3, 86, 33, 80], [56, 55, 34, 6, 25], [7, 55, 57, 81, 28, 95, 24, 23, 40], [54, 93, 99, 82, 33, 68, 45, 40], [3, 42, 20, 91, 41, 38, 68, 4, 19], [83, 61, 44, 39, 69, 91, 4, 28, 86], [74, 33, 28, 95, 55, 61, 21, 78, 69, 24, 53, 43], [61, 82, 91, 88, 32, 43, 40, 38], [52, 5, 48, 94, 75, 28], [56, 86, 28, 33, 52, 35, 91, 20], [45, 97, 79, 7, 56, 52, 61, 86], [98, 74, 96, 35, 86, 39, 6, 55, 57], [82, 23, 60, 86, 28, 94, 92, 72], [30, 78, 3, 54, 38, 99, 68, 19, 41, 80, 43], [49, 42, 35, 25, 87, 7, 99], [41, 34, 25, 44, 89, 99, 61, 68], [57, 25, 52, 5, 23], [30, 40, 61, 55, 98], [55, 35, 3, 33, 49, 24, 60, 91, 96, 94], [52, 5, 57, 83, 72, 39], [23, 24, 72, 75, 31], [93, 99, 4, 6, 29, 57, 35, 82, 75, 39], [32, 79, 5, 90, 2, 39, 97, 86, 48], [89, 28, 79, 91, 23, 19, 31, 73, 80], [34, 21, 83, 29, 75, 80], [78, 57, 4, 55, 42, 38, 80, 83, 90, 20], [86, 78, 2, 22, 61, 48, 39, 92], [79, 83, 21, 22, 68], [21, 96, 83, 74, 78, 22, 18], [31, 83, 96, 89, 3, 55, 75], [19, 49, 95, 61, 83, 3, 7, 78, 38, 99], [82, 25, 28, 86, 42, 98, 41, 49, 4, 30, 89, 54], [81, 23, 78, 32, 31, 4, 38], [3, 74, 43, 55, 88, 80], [44, 80, 6, 22, 91], [53, 93, 6, 25, 96, 42, 3, 83], [32, 40, 18, 20, 45], [33, 53, 93, 60, 39, 25], [40, 54, 93, 97, 31, 2, 20, 95], [6, 24, 35, 97, 48, 80, 42, 57, 99, 31, 89, 18], [56, 43, 95, 93, 34, 7, 20], [31, 23, 78, 98, 45, 39, 7, 75], [30, 90, 43, 48, 75, 45, 69], [87, 92, 52, 32, 97, 20, 82, 99, 38, 78, 91, 7], [57, 24, 91, 49, 73, 7, 44, 99], [52, 21, 41, 79, 96, 48, 38, 83], [29, 48, 95, 19, 61, 23, 73, 91, 53], [73, 3, 74, 32, 20, 68, 89], [90, 21, 94, 83, 25, 48, 7, 45], [40, 38, 49, 74, 20, 60], [91, 82, 88, 92, 48, 20, 78, 38], [81, 60, 56, 53, 91, 92], [74, 49, 52, 2, 24, 38, 60, 94, 5], [4, 94, 83, 32, 80, 55], [81, 72, 55, 21, 42, 32, 82, 18], [89, 44, 24, 69, 98, 20, 39, 55, 81, 33, 28, 57, 73, 34, 92], [20, 29, 83, 56, 38, 75], [29, 20, 3, 52, 45, 80, 18, 69, 95], [42, 61, 87, 72, 91, 99, 74], [38, 69, 95, 90, 93, 79, 35, 6, 20], [75, 33, 69, 57, 22, 92, 31, 45, 41, 42, 5], [93, 94, 43, 40, 61, 30, 80], [41, 52, 68, 2, 56], [40, 3, 88, 7, 48, 54, 60, 91, 42, 73, 44, 31, 81, 33], [2, 98, 43, 56, 4, 53, 38, 30, 80], [45, 39, 40], [97, 56, 2, 5], [83, 21, 33, 68, 39, 78, 2], [40, 89, 43, 28, 44, 32, 35, 90], [80, 73, 99, 56, 6], [32, 43, 90, 29, 87], [81, 6, 61, 5, 35, 19, 75], [98, 19, 35, 40, 90, 73, 53, 79, 31], [34, 3, 90, 6, 93, 49, 39, 21, 5], [80, 25, 69, 38, 79, 20], [52, 34, 30, 21, 93, 54, 97, 88, 29, 32, 60], [4, 87, 95, 90, 96, 29], [21, 40, 75, 87, 55, 2, 82, 90], [25, 80, 89, 82, 91], [7, 41, 38, 44, 4, 34, 88, 2, 61, 91], [4, 31, 7, 79, 55, 95, 25], [90, 73, 28, 35, 3, 96, 81, 56, 61, 4, 23], [32, 35, 89, 54, 56], [38, 34, 2, 29, 95, 45, 87, 79, 54, 52, 61], [41, 19, 7, 45, 87, 95, 43, 39], [19, 90, 69, 96, 24, 45, 57, 23, 3, 82, 29, 61], [34, 19, 22, 7, 99, 3, 42, 41, 89, 4, 33, 79], [2, 5, 98, 56, 54], [80, 34, 29, 39, 94, 44, 40, 2, 19], [2, 82, 52, 5, 79, 30, 23], [88, 91, 7, 5], [74, 52, 57, 41, 25, 28, 43, 97, 35, 88, 55], [56, 33, 60, 96, 86, 92, 29, 18, 78, 88, 72], [21, 25, 31], [55, 98, 19, 28, 96, 4, 57, 41, 42, 34, 86], [38, 44, 99, 53, 60], [3, 38, 82, 75, 20, 94, 73, 33, 54], [68, 99, 88, 44, 43], [25, 74, 53, 43, 48, 55, 34, 45], [23, 98, 74, 28, 33, 19, 5, 88, 38, 6, 48], [18, 41, 60, 35, 91, 52, 56, 96, 45, 24], [89, 56, 69, 98, 3, 91, 81, 25], [90, 80, 20, 24, 45, 38, 48, 61, 4, 89, 35, 29, 23, 69], [40, 49, 34, 2, 92], [44, 6, 49, 81, 68, 39], [38, 40, 69, 48, 19, 86, 30, 35, 22, 20], [72, 21, 39, 87, 29, 89], [98, 20, 79, 3, 56, 23, 74, 54], [87, 89, 75, 18, 92, 56, 61, 32], [98, 87, 73, 23, 79, 68, 38, 97, 56], [48, 72, 60, 18, 30, 78], [7, 94, 25, 35, 22, 73, 56, 39], [52, 24, 41, 90, 94, 42, 30, 7, 33, 93, 74, 22, 2, 55, 97, 99, 82, 86], [25, 49, 22, 19, 94, 28], [73, 44, 3, 40, 68, 42, 80], [6, 3, 52, 49, 4, 72, 43, 39], [78, 23, 72, 41, 61, 35, 30, 38, 32, 80, 56], [28, 88, 82, 57, 31, 73, 99, 80, 40, 35, 74], [60, 20, 97, 56, 55, 92], [7, 97, 4, 52, 31, 69, 72, 55, 57], [35, 49, 98, 80, 4, 88, 22, 7, 78, 20], [40, 60, 33, 83, 4, 53, 92, 80], [56, 48, 95, 82, 60, 91, 25], [34, 2, 69, 22, 32, 39, 56, 83, 21], [73, 45, 78, 97, 23, 38, 20, 35, 95], [56, 53, 38, 23, 34, 20], [53, 33, 82, 34, 90, 97, 93, 79], [99, 54, 7, 29, 97, 74, 49, 44, 2, 78], [24, 4, 97, 68, 23, 52, 30, 57, 81, 42], [93, 53, 6, 54, 79, 23, 82, 45], [35, 97, 38, 30, 88, 48, 4, 73, 2, 54, 19], [44, 2, 23, 82, 88, 80, 73, 96], [81, 56, 90, 73, 79, 98, 39, 42, 53, 49, 74, 45, 89], [57, 31, 73, 23, 19, 89, 99, 35, 53], [42, 34, 41, 31, 49, 88, 75], [7, 4, 25, 32, 68], [34, 18, 4, 89, 6, 54, 49], [69, 40, 39, 3, 56, 19, 92, 7, 20], [42, 60, 24, 5, 6, 28, 68], [98, 89, 73, 93, 41, 68, 29, 78, 31], [38, 18, 5, 80], [54, 98, 57, 73, 49, 31], [95, 28, 91, 24, 35, 42, 7, 92, 75, 99], [89, 41, 97, 53, 49, 54, 72], [54, 57, 89, 25, 69, 19, 82], [78, 92, 21, 45, 97, 41, 33, 81, 98, 57], [31, 90, 68, 33, 93, 49, 44], [34, 20, 83, 69, 55, 38, 52], [43, 4, 68, 61, 73], [82, 42, 33, 55, 92, 25, 34], [61, 94, 39, 31, 98, 2, 55, 44, 19, 52], [33, 93, 39, 30, 25], [98, 95, 87, 19, 48, 30, 25, 68, 34], [83, 96, 5, 95, 54, 18], [7, 40, 31, 39, 48, 68], [34, 98, 40, 19, 55, 45, 96, 92], [83, 97, 72, 49, 7, 60, 33, 44], [56, 96, 94, 82, 73, 48, 52, 87, 89, 93, 78, 75, 61], [68, 60, 45, 41, 32, 25, 74, 31], [55, 41, 34, 23, 78, 24, 81, 90, 5], [68, 86, 91, 23, 19, 39], [18, 61, 21, 54, 78, 99, 4, 44, 86, 39, 7, 49, 57], [81, 38, 23, 31], [32, 28, 41, 89, 55, 24, 93, 95, 6, 69, 53, 5], [73, 97, 48, 54, 30, 7, 61, 95], [96, 5, 54, 6, 49], [56, 55, 83, 23, 31, 6, 38, 68, 45, 19, 87, 35], [20, 81, 18, 39, 41, 54, 3]]

    #for lclause in leared_clauses:
    #    division_solver.add_learnt_clause(clause=lclause, clause_in_DIMACS=False)
    #sat = division_solver.solve()
    #set_stat(file_location, "division_solver_jump_forward", sat, (time.perf_counter() - start), num_vars, num_clauses)
    #print(f"Division solvable: {sat}. Solved in {1000 * (time.perf_counter() - start):.2f}ms")
    #if not sat:
    #    print(f"Trails: {division_solver.trails}")
    #    print(f"clausesA {clauses_A}")
    #    print(f"ClausesB: {clauses_B}")
    #    print(f"Glue: {glue_variables}")
    #    print(f"learnt clauses: {division_solver.learnt_clauses}")

    #assert sat == sat1
    #print(f"Variables: {division_solver.glue_variables}")
    #print(f"Tails: {division_solver.trails}")


for _ in range(0):
    create_split_formula("../instances/random_sat/uf50-048.cnf", False)


max = 10
for entry in os.scandir("../instances/random_sat"):
    max -= 1
    if (max < 0):
        break
    print(f"Next: {entry.name}")
    create_split_formula(entry.path, False)

#save_stats()

In [ ]:
#cnf = CNF(from_clauses=[[-1, 2], [-1, -2]])
cnf = CNF(from_file=cnf_file_location)

# create a SAT solver for this formula:
with Kissat404(bootstrap_with=cnf) as solver:
    print(solver.solve())
    #print(solver)
    #print(solver.get_proof())